In [0]:
from __future__ import annotations
import requests
import urllib3
import pandas as pd
import logging
import os
import time
import json
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, BooleanType
from datetime import datetime
import numpy as np
import re
import io
import base64
from pyspark.sql.functions import lit


In [0]:
%sql
    -- select universe_id, universe_name
    -- from (
    --     select *,
    --         row_number() over (partition by universe_id order by ingestion_ts desc) as rn
    --     from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_extraction_summary
    -- ) t
    -- where rn = 1 and ingestion_ts is not null
select universe_name, count(distinct DataItem_unique_Id) as cnt from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_datafields_API_list group by universe_name order by cnt desc


In [0]:


list_of_universes_df = spark.sql("""
    select universe_id, universe_name
    from (
        select *,
            row_number() over (partition by universe_id order by ingestion_ts desc) as rn
        from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_extraction_summary
    ) t
    where rn = 1 and ingestion_ts is null
""")
display(list_of_universes_df)

In [0]:
user = "ESBWIL1"
password = "Built2@escape"

# # Dev Configurations
# bo_url = "https://bobidev.atradiusnet.com:8443/biprws/"
# base_path = "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/dev"
# parameter_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_parameters"
# variable_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_variables"
# sql_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.Dev_webi_metadata_cms"
# extract_summary_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.dev_webi_document_extraction_summary"

# Prod files
bo_url = "https://wavgbcdfp1088.atradiusnet.com:8443/biprws/"
base_path = "/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi/prod"
universe_API_Table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_datafields_API_list"
universe_IDT_Table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_unvx_definitions"
extract_summary_table = "dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_extraction_summary"


In [0]:
class SAP_BO_Connection:
    def __init__(self, bo_url, user, password, retrial:int= 3):
        self.user = user
        self.password = password
        self.bo_url = bo_url
        self.logon_status = False
        self.retrial=retrial
        self.conection_timeout=5
        self.sleeptime=5
        self.logonToken = self.logon_request()
        self.headers = {
            "Content-Type": "application/json",
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0",
            "X-SAP-LogonToken": self.logonToken,
        }
        self.pdf_headers = {
            "Content-Type": "application/pdf",
            "Accept": "application/pdf",
            "User-Agent": "Mozilla/5.0",
            "X-SAP-LogonToken": self.logonToken,
        }
        self.html_headers = {
            "Content-Type": "text/html",
            "User-Agent": "Mozilla/5.0",
            "X-SAP-LogonToken": self.logonToken,
        }

    def logon_request(self):
        logon_url = f"{self.bo_url}/logon/long"
        auth_type = "secLDAP"
        logon_payload = {
            "userName": self.user,
            "password": self.password,
            "auth": auth_type,
        }
        headers = {"Content-Type": "application/json"}
        print("Logging into BO REST API...")
        try:
            response = requests.post(
                logon_url, json=logon_payload, headers=headers, verify=False
            )
            logon_token = response.headers.get("X-SAP-LogonToken", None)
            print(f"\tLogon Token: {logon_token}\n")
            self.logon_status = True if logon_token else False
        except Exception as e:
            print(
                "Failed to logon SAP BO Server:",
                logon_url,
                " with error: ",
                e,
                "exiting script.",
            )
            self.logon_status = False
            logon_token = None
        return logon_token

    def logoff_request(self):
        if self.logon_status is True:
            logoff_url = f"{self.bo_url}/logoff"
            logoff_headers = {"X-SAP-LogonToken": self.logonToken}
            requests.post(logoff_url, headers=logoff_headers, verify=False)
            self.logon_status = False
            print("Logged off from BO REST API.")
            return True
        else:
            raise Exception("Not a valid logon token")
            return False

    def get_universe(self, universe_id):
        if self.logon_status is True:
            universe_url = f"{self.bo_url}raylight/v1/universes/{universe_id}"
            try:
                i=1
                universe_response = None
                print(f"fetching universe data from {universe_url}")
                while i<=self.retrial and (universe_response is None or universe_response.status_code!=200):
                    universe_response = requests.get(
                        universe_url, headers=self.headers, timeout=(self.conection_timeout, 100), verify=False
                    )
                    i+=1
                    if universe_response.status_code==200:
                        universe_data = universe_response.json()
                    elif universe_response.status_code==401:
                        raise Exception(universe_response.json().get("message"))
                    elif universe_response.status_code==404:
                        raise Exception(universe_response.json().get("message"))
                    else:
                        time.sleep(self.sleeptime)
                
                return universe_data
            except Exception as e:
                print(
                    "Failed to retrieve universe data:",
                    universe_url,
                    "with error:",
                    e,
                    "exiting call.",
                )
                raise Exception(e)
        else:
            raise Exception("Not a valid logon token")
            return None


In [0]:

class Universe:
    def __init__(self, name, id):
        self.name: str = name
        self.id = id
        self.type: str = None
        self.path: str = None
        self.data_items: list = []

    def extract_items_from_folder(self, folder, parent_folder=None):
        rows = []
        folder_name = folder.get("name", parent_folder)
        items = folder.get("item", [])
        for item in items:
            row = {
                "universe_id": self.id,
                "universe_name": self.name,
                "universe_type": self.type,
                "universe_path": self.path,
                "folder_name": folder_name
            }
            for k, v in item.items():
                if k.startswith("@"):
                    row[f"DataItem_{k[1:]}"] = v
                else:
                    row[f"DataItem_{k}"] = v
            rows.append(row)
        # Recursively process subfolders if present
        for subfolder in folder.get("folder", []):
            rows.extend(self.extract_items_from_folder(subfolder, subfolder.get("name", folder_name)))
        return rows

    def extract_all_items(self, data):
        universe = data["universe"]
        self.id = universe["id"]
        self.name = universe["name"]
        self.type = universe["type"]
        self.path = universe["path"]
        outline = universe.get("outline", {})
        folders = outline.get("folder", [])
        self.data_items = []
        for folder in folders:
            self.data_items.extend(self.extract_items_from_folder(folder))
        return len(self.data_items)
    def summary(self):
        return {
            "universe_id":self.id,
            "universe_name":self.name,
            "universe_type":self.type,
            "data_items":len(self.data_items),
            "ingestion_ts":datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }

# Example usage:


In [0]:

from pyspark.sql import functions as F

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
bo_connection = SAP_BO_Connection(bo_url, user, password)
if not bo_connection.logon_status:
    print("Failed to logon. Exiting.")
    raise SystemExit("Logon failed")

# Load target table schema once to enforce consistent types on every write
target_schema = spark.table(universe_API_Table).schema
target_col_names = [f.name for f in target_schema.fields]

try:
    for universe in list_of_universes_df.take(200):
        universe_sample = Universe(name=universe.universe_name, id=universe.universe_id)
        universe_data = bo_connection.get_universe(universe.universe_id)
        universe_sample.extract_all_items(universe_data)
        # Convert complex nested fields (lists/dicts) to JSON strings for Spark compatibility
        flattened_items = []
        for item in universe_sample.data_items:
            row = {}
            for k, v in item.items():
                if isinstance(v, (list, dict)):
                    row[k] = json.dumps(v)
                else:
                    row[k] = str(v) if v is not None else None
            row["DataItem_unique_Id"] = f"{row.get('universe_id', '')}_{row.get('DataItem_id', '')}"
            flattened_items.append(row)

        data_item_df = spark.createDataFrame(flattened_items)

        # Align DataFrame to UC table schema:
        # add missing columns as null, cast existing columns to target types, select in table order
        for field in target_schema.fields:
            if field.name not in data_item_df.columns:
                data_item_df = data_item_df.withColumn(field.name, F.lit(None).cast(field.dataType))
            else:
                data_item_df = data_item_df.withColumn(field.name, F.col(field.name).cast(field.dataType))
        data_item_df = data_item_df.select(target_col_names)

        # display(data_item_df)
        data_item_df.write.mode("append").saveAsTable(universe_API_Table)
        summary_df = spark.createDataFrame([universe_sample.summary()])
        # display(summary_df)
        summary_df.write.mode("append").saveAsTable(extract_summary_table)
        print(f"Universe {universe.universe_name} processed.")
except Exception as e:
    print("Error:", e)

bo_connection.logoff_request()


In [0]:
%sql
select * from  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_extraction_summary
-- where universe_id='19425656' and ingestion_ts is null

In [0]:
%sql
select  universe_name, case when Ingested_ts is null then 'Before' else 'Now' end as Refreshed, count(*) as rows_cnt 
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.universe_unvx_definitions
group by universe_name, Refreshed
order by universe_name asc


In [0]:
File_name="/Workspace/Users/baodi.wilkinson.external@atradius.com/SAP_BO_Analytics/Universe Data/UNX BLX files/Ingested/Workflow.txt"

In [0]:
# Read the BLX txt file as a single string
with open(File_name, "r", encoding="cp1252") as f:
    blx_content = f.read()

import re
import json

# Parse BLX into hierarchical JSON
# Rules:
#   "-------------------------------------------------" is the delimiter
#   Every 4 leading spaces before the delimiter = one level down
#   Same indentation level = sibling sections under the same parent

lines = blx_content.splitlines()
delimiter_pattern = re.compile(r'^(\s*)-{10,}\s*$')

# Pass 1: Identify all delimiters, their levels, and their section headers
section_markers = []
for i, line in enumerate(lines):
    match = delimiter_pattern.match(line)
    if match:
        indent = len(match.group(1))
        level = indent // 4
        # Look backwards for the section header (first non-empty line before delimiter)
        header = ""
        for j in range(i - 1, -1, -1):
            stripped = lines[j].strip()
            if stripped and not delimiter_pattern.match(lines[j]):
                header = stripped
                break
        section_markers.append({
            "line_index": i,
            "level": level,
            "header": header,
            "content_start": i + 1
        })

# Pass 2: Determine content boundaries for each section
for idx in range(len(section_markers)):
    current_level = section_markers[idx]["level"]
    content_start = section_markers[idx]["content_start"]
    content_end = len(lines)  # default: until end of file

    # Content ends where the next section at the same or higher (lower number) level begins
    for next_idx in range(idx + 1, len(section_markers)):
        if section_markers[next_idx]["level"] <= current_level:
            # Exclude the header line of the next section from our content
            content_end = section_markers[next_idx]["line_index"]
            for j in range(content_end - 1, content_start - 1, -1):
                if lines[j].strip() and not delimiter_pattern.match(lines[j]):
                    content_end = j  # stop before the next header
                    break
            break

    # Extract raw content lines (excluding child section headers/delimiters)
    raw_content_lines = []
    child_start_lines = {m["line_index"] for m in section_markers if m["line_index"] > section_markers[idx]["line_index"] and m["line_index"] < content_end}
    # Get header lines of children so we can exclude them from content
    child_header_lines = set()
    for m in section_markers:
        if m["line_index"] > section_markers[idx]["line_index"] and m["line_index"] <= content_end and m["level"] > current_level:
            # Find its header line
            for j in range(m["line_index"] - 1, content_start - 1, -1):
                stripped = lines[j].strip()
                if stripped and not delimiter_pattern.match(lines[j]):
                    child_header_lines.add(j)
                    break
            child_start_lines.add(m["line_index"])

    for li in range(content_start, content_end):
        if li not in child_start_lines and li not in child_header_lines:
            raw_content_lines.append(lines[li])

    section_markers[idx]["content"] = "\n".join(raw_content_lines).strip()

# Pass 3: Build hierarchical JSON using a stack
def build_hierarchy(markers):
    root = {"name": "root", "children": []}
    stack = [(-1, root)]  # (level, node)

    for marker in markers:
        node = {
            "name": marker["header"],
            "level": marker["level"],
            "content": marker["content"],
            "children": []
        }
        # Pop stack until we find a parent (level < current)
        while len(stack) > 1 and stack[-1][0] >= marker["level"]:
            stack.pop()
        # Attach as child of the current stack top
        stack[-1][1]["children"].append(node)
        stack.append((marker["level"], node))

    return root

blx_json = build_hierarchy(section_markers)

# Helper to search nodes by name prefix
def find_nodes_by_prefix(node, prefix, results=None):
    if results is None:
        results = []
    if node.get("name", "").startswith(prefix):
        results.append(node)
    for child in node.get("children", []):
        find_nodes_by_prefix(child, prefix, results)
    return results

# Extract universe_name from "General Information" node
gen_info_node = next((n for n in find_nodes_by_prefix(blx_json, "General Information") if n["name"] == "General Information"), None)
blx_universe_name = ""
if gen_info_node:
    for line in gen_info_node.get("content", "").splitlines():
        line = line.strip()
        if line.startswith("Name :"):
            blx_universe_name = line.split(" : ", 1)[1].strip()
            break

print(f"Universe Name: {blx_universe_name}")

# Preview the structure
# print(json.dumps(blx_json, indent=2)[:3000])

In [0]:
# Reusable helper: parse BLX key-value content, correctly handling multi-line values
def parse_kv_content(text, prefix=""):
    """Parse BLX key-value content lines into a dict.
    Lines that do not start a new key are treated as continuations of the previous value,
    fixing truncated multi-line SQL expressions like 'Select : sum (\n    TABLE.COL\n)'.
    """
    result = {}
    current_key = None
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            current_key = None  # blank line ends any continuation
            continue
        if " : " in stripped:
            key, value = stripped.split(" : ", 1)
            current_key = (f"{prefix}_{key.strip()}" if prefix else key.strip())
            result[current_key] = value.strip()
        elif stripped.endswith(" :"):
            current_key = (f"{prefix}_{stripped[:-2].strip()}" if prefix else stripped[:-2].strip())
            result[current_key] = ""
        elif current_key is not None:
            # Continuation of a multi-line value (e.g. multi-line SQL expression)
            result[current_key] = result[current_key] + " " + stripped
    return result

# Parse the "Folders" node content into individual records
folder_node = next(n for n in find_nodes_by_prefix(blx_json, "Folders") if n["name"] == "Folders")

# Split content into blocks by blank lines
content = folder_node["content"]
blocks = re.split(r'\n\s*\n', content)
blocks = [b.strip() for b in blocks if b.strip()]

# Parse each block using the multi-line-aware helper
folder_records = []
for block in blocks:
    record = parse_kv_content(block)
    if record:
        record["universe_name"] = blx_universe_name
        folder_records.append(record)

print(f"Parsed {len(folder_records)} folder records")

# Convert to Spark DataFrame
folder_df = spark.createDataFrame(folder_records)
# display(folder_df)

In [0]:
# Parse "Dimensions and Attributes" node
# Children are named "Dimension: <name>" or "Attribute: <name>"
# Each child has sub-nodes "SQL Definition" and "Advanced" with key-value content

dim_attr_parent = next((n for n in find_nodes_by_prefix(blx_json, "Dimensions and Attributes") if n["name"] == "Dimensions and Attributes"), None)
print(f"Found 'Dimensions and Attributes' node: {dim_attr_parent is not None}")

def parse_dim_attr_children(parent_node):
    """Parse child nodes of 'Dimensions and Attributes' into records."""
    records = []
    if parent_node is None:
        return records
    for child in parent_node.get("children", []):
        name = child.get("name", "")
        # Determine record type from prefix
        if name.startswith("Dimension:"):
            record_type = "Dimension"
            item_name = name[len("Dimension:"):].strip()
        elif name.startswith("Attribute:"):
            record_type = "Attribute"
            item_name = name[len("Attribute:"):].strip()
        else:
            continue

        record = {"record_type": record_type, "Name": item_name, "universe_name": blx_universe_name}

        # Parse the main content (multi-line-aware)
        record.update(parse_kv_content(child.get("content", "")))

        # Parse sub-children (SQL Definition, Advanced) with section name as prefix
        for sub_child in child.get("children", []):
            sub_name = sub_child.get("name", "")
            record.update(parse_kv_content(sub_child.get("content", ""), prefix=sub_name))

        records.append(record)
    return records

all_records = parse_dim_attr_children(dim_attr_parent)
dim_count = sum(1 for r in all_records if r["record_type"] == "Dimension")
attr_count = sum(1 for r in all_records if r["record_type"] == "Attribute")
print(f"Parsed {dim_count} Dimension records, {attr_count} Attribute records")

# Convert to Spark DataFrame
dim_attr_df = spark.createDataFrame(all_records)
# display(dim_attr_df)

In [0]:
# Find the parent node for Measures (could be "Measures", "Measures and Attributes", etc.)
measure_parent = None
for candidate in ["Measures", "Measures and Attributes"]:
    match = next((n for n in find_nodes_by_prefix(blx_json, candidate) if n["name"] == candidate), None)
    if match:
        measure_parent = match
        break

# If no exact match, search for any node whose children start with "Measure:"
if measure_parent is None:
    def find_measure_parent(node):
        for child in node.get("children", []):
            if child.get("name", "").startswith("Measure:"):
                return node
            result = find_measure_parent(child)
            if result:
                return result
        return None
    measure_parent = find_measure_parent(blx_json)

print(f"Measure parent node: {measure_parent['name'] if measure_parent else 'Not found'}")
print(f"Number of children: {len(measure_parent.get('children', [])) if measure_parent else 0}")

# Parse all Measure children
measure_records = []
if measure_parent:
    for child in measure_parent.get("children", []):
        name = child.get("name", "")
        if name.startswith("Measure:"):
            item_name = name[len("Measure:"):].strip()
        else:
            continue

        record = {"record_type": "Measure", "Name": item_name, "universe_name": blx_universe_name}

        # Parse main content (multi-line-aware)
        record.update(parse_kv_content(child.get("content", "")))

        # Parse sub-children (SQL Definition, Advanced, etc.) with section name as prefix
        for sub_child in child.get("children", []):
            sub_name = sub_child.get("name", "")
            record.update(parse_kv_content(sub_child.get("content", ""), prefix=sub_name))

        measure_records.append(record)

print(f"Parsed {len(measure_records)} Measure records")

# Guard against empty list — some universes have no Measures
if measure_records:
    measure_df = spark.createDataFrame(measure_records)
    # display(measure_df)
else:
    print("WARNING: No Measure records found — measure_df set to None")
    measure_df = None

In [0]:
# Find the parent node for Filters
filter_parent = None
for candidate in ["Filters", "Filters and Attributes"]:
    match = next((n for n in find_nodes_by_prefix(blx_json, candidate) if n["name"] == candidate), None)
    if match:
        filter_parent = match
        break

# Fallback: search for any node whose children start with "Filter:"
if filter_parent is None:
    def find_filter_parent(node):
        for child in node.get("children", []):
            if child.get("name", "").startswith("Filter:"):
                return node
            result = find_filter_parent(child)
            if result:
                return result
        return None
    filter_parent = find_filter_parent(blx_json)

print(f"Filter parent node: {filter_parent['name'] if filter_parent else 'Not found'}")
print(f"Number of children: {len(filter_parent.get('children', [])) if filter_parent else 0}")

# Parse all Filter children
filter_records = []
if filter_parent:
    for child in filter_parent.get("children", []):
        name = child.get("name", "")
        if name.startswith("Filter:"):
            item_name = name[len("Filter:"):].strip()
        else:
            continue

        record = {"record_type": "Filter", "Name": item_name, "universe_name": blx_universe_name}

        # Parse main content (multi-line-aware)
        record.update(parse_kv_content(child.get("content", "")))

        # Parse sub-children (SQL Definition, Advanced, etc.) with section name as prefix
        for sub_child in child.get("children", []):
            sub_name = sub_child.get("name", "")
            record.update(parse_kv_content(sub_child.get("content", ""), prefix=sub_name))

        filter_records.append(record)

print(f"Parsed {len(filter_records)} Filter records")

# Guard against empty list — some universes have no Filters
if filter_records:
    filter_df = spark.createDataFrame(filter_records)
    # display(filter_df)
else:
    print("WARNING: No Filter records found — filter_df set to None")
    filter_df = None

In [0]:
from pyspark.sql.functions import lit, current_timestamp

# Add record_type to folder_df since it doesn't have one
folder_df_typed = folder_df.withColumn("record_type", lit("Folder"))

# Merge all DataFrames using unionByName with allowMissingColumns
# measure_df / filter_df may be None when a universe has no Measures or Filters
blx_combined_df = folder_df_typed.unionByName(dim_attr_df, allowMissingColumns=True)
if measure_df is not None:
    blx_combined_df = blx_combined_df.unionByName(measure_df, allowMissingColumns=True)
else:
    print("Skipping measure_df union (no measures in this universe)")
if filter_df is not None:
    blx_combined_df = blx_combined_df.unionByName(filter_df, allowMissingColumns=True)
else:
    print("Skipping filter_df union (no filters in this universe)")
# Sanitize column names: replace spaces and special characters with underscores
import re as _re
def sanitize_col_name(name):
    return _re.sub(r'[ ,;:{}()\n\t=]+', '_', name).strip('_')

blx_combined_df = blx_combined_df.toDF(*[sanitize_col_name(c) for c in blx_combined_df.columns])

# Reorder columns: priority columns first, then everything else
priority_cols = ["universe_name", "record_type", "Cuid", "Name", "State", "Data_Type", "Description", "SQL_Definition_Select"]
remaining_cols = [c for c in blx_combined_df.columns if c not in priority_cols]
blx_combined_df = blx_combined_df.select(priority_cols + remaining_cols)


if "Inherited_from_Core" not in blx_combined_df.columns:
    blx_combined_df = blx_combined_df.withColumn("Inherited_from_Core", lit(None).cast("string"))

# Stamp the ingestion time — matches the existing 'Ingested_ts' column in the target table
blx_combined_df = blx_combined_df.withColumn("Ingested_ts", current_timestamp())

print(f"Combined DataFrame: {blx_combined_df.count()} rows, {len(blx_combined_df.columns)} columns")
# display(blx_combined_df)

# Only write columns that exist in the target table
target_columns = [col.name for col in spark.table(universe_IDT_Table).schema]
matching_cols = [c for c in target_columns if c in blx_combined_df.columns]
dropped_cols = [c for c in blx_combined_df.columns if c not in target_columns]
print(f"Writing {len(matching_cols)} columns, dropping {len(dropped_cols)}: {dropped_cols}")

# Delete existing records for this universe before writing to avoid duplicates on re-runs
spark.sql(f"DELETE FROM {universe_IDT_Table} WHERE universe_name = '{blx_universe_name}'")
print(f"Deleted existing records for universe: {blx_universe_name}")

blx_combined_df.select(matching_cols).write.mode("append").saveAsTable(universe_IDT_Table)
# blx_combined_df.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(universe_IDT_Table)